<a href="https://colab.research.google.com/github/goitstudent123/llm-gen/blob/main/dz_final_has.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Chunk 1: Install dependencies
!pip -q install -U pandas openpyxl requests tqdm langchain langchain-openai
!pip -q check

In [2]:
# Chunk 2: Initial imports and OpenRouter setup
import os
import re
import json
import math
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI


def read_colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_api_key = (
    read_colab_secret("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_API_KEY")
)

OPENROUTER_MODEL = (
    read_colab_secret("OPENROUTER_MODEL")
    or os.getenv("OPENROUTER_MODEL")
    or "openai/gpt-4.1-mini"
)

OPENROUTER_HTTP_REFERER = (
    read_colab_secret("OPENROUTER_HTTP_REFERER")
    or os.getenv("OPENROUTER_HTTP_REFERER")
    or "https://colab.research.google.com"
)

OPENROUTER_APP_TITLE = (
    read_colab_secret("OPENROUTER_APP_TITLE")
    or os.getenv("OPENROUTER_APP_TITLE")
    or "Colab Excel Enrichment"
)

llm = None

if openrouter_api_key:
    llm = ChatOpenAI(
        model=OPENROUTER_MODEL,
        api_key=openrouter_api_key,
        base_url=OPENROUTER_BASE_URL,
        temperature=0,
        default_headers={
            "HTTP-Referer": OPENROUTER_HTTP_REFERER,
            "X-OpenRouter-Title": OPENROUTER_APP_TITLE,
        },
    )
    print(f"OpenRouter model: {OPENROUTER_MODEL}")
else:
    print("OPENROUTER_API_KEY was not found. The deterministic planner will be used.")

OpenRouter model: openai/gpt-4.1-mini


In [3]:
# Chunk 3: Colab file upload and Google Sheets download helpers
def upload_excel_files() -> List[str]:
    try:
        from google.colab import files
        uploaded = files.upload()
        return list(uploaded.keys())
    except Exception as error:
        print(f"Upload is available only in Colab: {error}")
        return []


def download_result_file(file_path: str) -> None:
    try:
        from google.colab import files
        files.download(file_path)
    except Exception as error:
        print(f"Download is available only in Colab: {error}")


def extract_google_sheet_id(url: str) -> str:
    match = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", url)
    if not match:
        raise ValueError("Could not extract Google Sheet ID from URL.")
    return match.group(1)


def extract_google_sheet_gid(url: str) -> str:
    match = re.search(r"gid=([0-9]+)", url)
    return match.group(1) if match else "0"


def download_google_sheet_as_xlsx(url: str, output_path: str) -> str:
    sheet_id = extract_google_sheet_id(url)
    gid = extract_google_sheet_gid(url)

    export_url = (
        f"https://docs.google.com/spreadsheets/d/{sheet_id}/export"
        f"?format=xlsx&gid={gid}"
    )

    response = requests.get(export_url, timeout=30)

    if response.status_code != 200:
        raise RuntimeError(
            f"Could not download Google Sheet. Status: {response.status_code}"
        )

    Path(output_path).write_bytes(response.content)
    return output_path

In [4]:
# Chunk 4: Create sample Excel files
def create_sample_excel_files() -> Tuple[str, str]:
    capitals_df = pd.DataFrame(
        [
            {
                "Capital_From": "Kyiv",
                "Country_From": "Ukraine",
                "Capital_To": "Paris",
                "Country_To": "France",
                "distance": None,
            },
            {
                "Capital_From": "London",
                "Country_From": "United Kingdom",
                "Capital_To": "Rome",
                "Country_To": "Italy",
                "distance": None,
            },
            {
                "Capital_From": "Paris",
                "Country_From": "France",
                "Capital_To": "Berlin",
                "Country_To": "Germany",
                "distance": None,
            },
            {
                "Capital_From": "Berlin",
                "Country_From": "Germany",
                "Capital_To": "Vienna",
                "Country_To": "Austria",
                "distance": None,
            },
            {
                "Capital_From": "Warsaw",
                "Country_From": "Poland",
                "Capital_To": "Kyiv",
                "Country_To": "Ukraine",
                "distance": None,
            },
        ]
    )

    mountains_df = pd.DataFrame(
        [
            {"Mountain": "Everest", "Country": "Nepal", "height": None},
            {"Mountain": "Mont Blanc", "Country": "France", "height": None},
            {"Mountain": "Denali", "Country": "USA", "height": None},
            {"Mountain": "Kilimanjaro", "Country": "Tanzania", "height": None},
        ]
    )

    capitals_path = "capitals.xlsx"
    mountains_path = "mountains.xlsx"

    capitals_df.to_excel(capitals_path, index=False)
    mountains_df.to_excel(mountains_path, index=False)

    return capitals_path, mountains_path


capitals_path, mountains_path = create_sample_excel_files()

print(capitals_path)
print(mountains_path)

capitals.xlsx
mountains.xlsx


In [5]:
# Chunk 5: Analyze task and build enrichment plan
def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def find_column_case_insensitive(columns: List[str], expected_name: str) -> Optional[str]:
    expected_lower = expected_name.lower()

    for column in columns:
        if column.lower() == expected_lower:
            return column

    return None


def deterministic_plan(df: pd.DataFrame, task_description: str) -> Dict[str, Any]:
    columns = list(df.columns)
    text = task_description.lower()

    distance_column = find_column_case_insensitive(columns, "distance")
    height_column = find_column_case_insensitive(columns, "height")

    if distance_column or "відстан" in text or "distance" in text:
        return {
            "task_type": "capital_distance_km",
            "target_column": distance_column or "distance",
            "required_columns": [
                "Capital_From",
                "Country_From",
                "Capital_To",
                "Country_To",
            ],
            "value_unit": "km",
        }

    if height_column or "висот" in text or "height" in text:
        return {
            "task_type": "mountain_height_m",
            "target_column": height_column or "height",
            "required_columns": [
                "Mountain",
                "Country",
            ],
            "value_unit": "m",
        }

    return {
        "task_type": "unsupported",
        "target_column": "",
        "required_columns": [],
        "value_unit": "",
    }


def analyze_task(df: pd.DataFrame, task_description: str) -> Dict[str, Any]:
    fallback_plan = deterministic_plan(df, task_description)

    if llm is None:
        return fallback_plan

    sample_rows = df.head(5).fillna("").to_dict(orient="records")

    prompt = f"""
You analyze Excel enrichment tasks.

Return JSON only with this schema:
{{
  "task_type": "capital_distance_km|mountain_height_m|unsupported",
  "target_column": "column name to fill or create",
  "required_columns": ["column names needed from the file"],
  "value_unit": "km|m|"
}}

Task description:
{task_description}

Columns:
{list(df.columns)}

Sample rows:
{sample_rows}
""".strip()

    try:
        response = llm.invoke(prompt)
        parsed = extract_json_object(response.content)

        if not parsed:
            return fallback_plan

        if parsed.get("task_type") not in {
            "capital_distance_km",
            "mountain_height_m",
            "unsupported",
        }:
            return fallback_plan

        if not parsed.get("target_column"):
            return fallback_plan

        return parsed
    except Exception:
        return fallback_plan


def validate_required_columns(df: pd.DataFrame, required_columns: List[str]) -> None:
    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

In [6]:
# Chunk 6: Wikidata internet lookup helpers
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

HTTP_HEADERS = {
    "User-Agent": "colab-excel-enrichment/1.0 educational project"
}


def request_json(url: str, params: Dict[str, Any], timeout: int = 30) -> Dict[str, Any]:
    response = requests.get(
        url,
        params=params,
        headers=HTTP_HEADERS,
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()


def search_wikidata_entities(query: str, limit: int = 5) -> List[Dict[str, Any]]:
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": query,
        "limit": limit,
    }

    data = request_json(WIKIDATA_API_URL, params)
    return data.get("search", [])


def get_wikidata_entities(qids: List[str]) -> Dict[str, Any]:
    if not qids:
        return {}

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": "|".join(qids),
        "props": "claims|labels",
        "languages": "en",
    }

    data = request_json(WIKIDATA_API_URL, params)
    return data.get("entities", {})


def get_claim_value(entity: Dict[str, Any], property_id: str) -> Optional[Any]:
    claims = entity.get("claims", {}).get(property_id, [])

    if not claims:
        return None

    first_claim = claims[0].get("mainsnak", {})
    data_value = first_claim.get("datavalue", {})

    return data_value.get("value")


def extract_coordinates(entity: Dict[str, Any]) -> Optional[Tuple[float, float]]:
    value = get_claim_value(entity, "P625")

    if not value:
        return None

    return float(value["latitude"]), float(value["longitude"])


def extract_elevation_meters(entity: Dict[str, Any]) -> Optional[int]:
    value = get_claim_value(entity, "P2044")

    if not value:
        return None

    amount = float(value["amount"])
    return int(round(amount))


def entity_url(qid: str) -> str:
    return f"https://www.wikidata.org/wiki/{qid}"

In [15]:
# Chunk 7: Enrichment logic for distances and mountain heights
CAPITAL_COORDINATE_FALLBACK = {
    ("kyiv", "ukraine"): (50.4501, 30.5234),
    ("paris", "france"): (48.8566, 2.3522),
    ("london", "united kingdom"): (51.5074, -0.1278),
    ("rome", "italy"): (41.9028, 12.4964),
    ("berlin", "germany"): (52.5200, 13.4050),
    ("vienna", "austria"): (48.2082, 16.3738),
    ("warsaw", "poland"): (52.2297, 21.0122),
    ("madrid", "spain"): (40.4168, -3.7038),
    ("lisbon", "portugal"): (38.7223, -9.1393),
    ("stockholm", "sweden"): (59.3293, 18.0686),
    ("oslo", "norway"): (59.9139, 10.7522),
    ("budapest", "hungary"): (47.4979, 19.0402),
    ("athens", "greece"): (37.9838, 23.7275),
    ("istanbul", "turkey"): (41.0082, 28.9784),
}

MOUNTAIN_HEIGHT_FALLBACK = {
    "mount everest": 8849,
    "everest": 8849,
    "k2": 8611,
    "kangchenjunga": 8586,
    "lhotse": 8516,
    "makalu": 8485,
    "cho oyu": 8188,
    "dhaulagiri": 8167,
    "manaslu": 8163,
    "nanga parbat": 8126,
    "annapurna": 8091,
    "mont blanc": 4808,
    "denali": 6190,
    "kilimanjaro": 5895,
}


def normalize_string(value: Any) -> str:
    return str(value).strip().lower()


def normalized_key(*parts: Any) -> Tuple[str, ...]:
    return tuple(normalize_string(part) for part in parts)


def haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    earth_radius_km = 6371.0088

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    value = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(delta_lon / 2) ** 2
    )

    central_angle = 2 * math.atan2(math.sqrt(value), math.sqrt(1 - value))
    return earth_radius_km * central_angle


def resolve_capital_coordinates(capital: str, country: str) -> Dict[str, Any]:
    fallback_key = normalized_key(capital, country)
    fallback_coordinates = CAPITAL_COORDINATE_FALLBACK.get(fallback_key)

    if fallback_coordinates:
        return {
            "ok": True,
            "coordinates": fallback_coordinates,
            "source": "local_fallback_coordinates",
            "status": "fallback",
        }

    search_queries = [
        f"{capital} capital {country}",
        f"{capital} {country}",
        str(capital),
    ]

    for query in search_queries:
        try:
            search_results = search_wikidata_entities(query, limit=10)
            qids = [item["id"] for item in search_results if item.get("id")]
            entities = get_wikidata_entities(qids)

            for qid in qids:
                entity = entities.get(qid, {})
                coordinates = extract_coordinates(entity)

                if coordinates:
                    return {
                        "ok": True,
                        "coordinates": coordinates,
                        "source": entity_url(qid),
                        "status": "ok",
                    }
        except Exception:
            continue

    return {
        "ok": False,
        "coordinates": None,
        "source": "",
        "status": "not_found",
    }


def resolve_mountain_height(mountain: str, country: str) -> Dict[str, Any]:
    mountain_key = normalize_string(mountain)
    fallback_height = MOUNTAIN_HEIGHT_FALLBACK.get(mountain_key)

    if fallback_height:
        return {
            "ok": True,
            "height": fallback_height,
            "source": "local_fallback_height",
            "status": "fallback",
        }

    search_queries = [
        f"{mountain} mountain",
        f"{mountain} peak",
        str(mountain),
    ]

    for query in search_queries:
        try:
            search_results = search_wikidata_entities(query, limit=10)
            qids = [item["id"] for item in search_results if item.get("id")]
            entities = get_wikidata_entities(qids)

            for qid in qids:
                entity = entities.get(qid, {})
                elevation = extract_elevation_meters(entity)

                if elevation:
                    return {
                        "ok": True,
                        "height": elevation,
                        "source": entity_url(qid),
                        "status": "ok",
                    }
        except Exception:
            continue

    return {
        "ok": False,
        "height": None,
        "source": "",
        "status": "not_found",
    }

In [16]:
# Chunk 8: Excel processing and saving functions
def is_empty_value(value: Any) -> bool:
    if pd.isna(value):
        return True

    return str(value).strip() == ""


def format_excel_file(file_path: str) -> None:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    workbook = load_workbook(file_path)
    worksheet = workbook.active

    header_fill = PatternFill("solid", fgColor="EAF2F8")

    for cell in worksheet[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")

    worksheet.freeze_panes = "A2"

    for column_cells in worksheet.columns:
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in column_cells
        )
        column_letter = get_column_letter(column_cells[0].column)
        worksheet.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 45)

    workbook.save(file_path)


def enrich_capital_distances(
    df: pd.DataFrame,
    target_column: str,
    overwrite: bool,
    include_debug_columns: bool,
    max_workers: int,
) -> pd.DataFrame:
    validate_required_columns(
        df,
        ["Capital_From", "Country_From", "Capital_To", "Country_To"],
    )

    if target_column not in df.columns:
        df[target_column] = None

    source_column = f"{target_column}_source"
    status_column = f"{target_column}_status"

    if include_debug_columns:
        df[source_column] = ""
        df[status_column] = ""

    unique_places = set()

    for _, row in df.iterrows():
        unique_places.add((row["Capital_From"], row["Country_From"]))
        unique_places.add((row["Capital_To"], row["Country_To"]))

    coordinate_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(resolve_capital_coordinates, capital, country): (capital, country)
            for capital, country in unique_places
        }

        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Resolving capitals"):
            key = future_map[future]
            try:
                coordinate_results[key] = future.result()
            except Exception as error:
                coordinate_results[key] = {
                    "ok": False,
                    "coordinates": None,
                    "source": "",
                    "status": f"error: {error}",
                }

    for index, row in df.iterrows():
        current_value = row.get(target_column)

        if not overwrite and not is_empty_value(current_value):
            continue

        from_key = (row["Capital_From"], row["Country_From"])
        to_key = (row["Capital_To"], row["Country_To"])

        from_result = coordinate_results.get(from_key)
        to_result = coordinate_results.get(to_key)

        if not from_result or not to_result or not from_result["ok"] or not to_result["ok"]:
            df.at[index, target_column] = None

            if include_debug_columns:
                df.at[index, source_column] = ""
                df.at[index, status_column] = "not_found"

            continue

        lat1, lon1 = from_result["coordinates"]
        lat2, lon2 = to_result["coordinates"]

        distance_km = round(haversine_km(lat1, lon1, lat2, lon2))
        df.at[index, target_column] = distance_km

        if include_debug_columns:
            df.at[index, source_column] = f"{from_result['source']} | {to_result['source']}"
            df.at[index, status_column] = "ok"

    return df


def enrich_mountain_heights(
    df: pd.DataFrame,
    target_column: str,
    overwrite: bool,
    include_debug_columns: bool,
    max_workers: int,
) -> pd.DataFrame:
    validate_required_columns(df, ["Mountain", "Country"])

    if target_column not in df.columns:
        df[target_column] = None

    source_column = f"{target_column}_source"
    status_column = f"{target_column}_status"

    if include_debug_columns:
        df[source_column] = ""
        df[status_column] = ""

    unique_mountains = {
        (row["Mountain"], row["Country"])
        for _, row in df.iterrows()
    }

    height_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(resolve_mountain_height, mountain, country): (mountain, country)
            for mountain, country in unique_mountains
        }

        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Resolving mountains"):
            key = future_map[future]
            try:
                height_results[key] = future.result()
            except Exception as error:
                height_results[key] = {
                    "ok": False,
                    "height": None,
                    "source": "",
                    "status": f"error: {error}",
                }

    for index, row in df.iterrows():
        current_value = row.get(target_column)

        if not overwrite and not is_empty_value(current_value):
            continue

        key = (row["Mountain"], row["Country"])
        result = height_results.get(key)

        if not result or not result["ok"]:
            df.at[index, target_column] = None

            if include_debug_columns:
                df.at[index, source_column] = ""
                df.at[index, status_column] = "not_found"

            continue

        df.at[index, target_column] = result["height"]

        if include_debug_columns:
            df.at[index, source_column] = result["source"]
            df.at[index, status_column] = result["status"]

    return df


def process_excel(
    file_path: str,
    task_description: str,
    output_path: Optional[str] = None,
    overwrite: bool = True,
    include_debug_columns: bool = True,
    max_workers: int = 4,
) -> str:
    df = pd.read_excel(file_path)
    plan = analyze_task(df, task_description)

    print("Enrichment plan:")
    print(json.dumps(plan, ensure_ascii=False, indent=2))

    task_type = plan["task_type"]
    target_column = plan["target_column"]

    if task_type == "unsupported":
        raise ValueError("Unsupported enrichment task.")

    if task_type == "capital_distance_km":
        enriched_df = enrich_capital_distances(
            df=df,
            target_column=target_column,
            overwrite=overwrite,
            include_debug_columns=include_debug_columns,
            max_workers=max_workers,
        )
    elif task_type == "mountain_height_m":
        enriched_df = enrich_mountain_heights(
            df=df,
            target_column=target_column,
            overwrite=overwrite,
            include_debug_columns=include_debug_columns,
            max_workers=max_workers,
        )
    else:
        raise ValueError(f"Unsupported task type: {task_type}")

    if output_path is None:
        input_path = Path(file_path)
        output_path = str(input_path.with_name(f"{input_path.stem}_enriched.xlsx"))

    enriched_df.to_excel(output_path, index=False)
    format_excel_file(output_path)

    print(f"Saved: {output_path}")
    return output_path

In [17]:
# Chunk 9: Download correct input files and run enrichment
capitals_url = "https://docs.google.com/spreadsheets/d/1Bv_WFNzGG4ZL7-QQf8dSrnSEwRvUnhrh/edit?gid=28170114#gid=28170114"
mountains_url = "https://docs.google.com/spreadsheets/d/1UutRbt0yvinVpz3cQdAcRSlqPFmyVEXx/edit?gid=1262408660#gid=1262408660"

download_google_sheet_as_xlsx(capitals_url, "capitals.xlsx")
download_google_sheet_as_xlsx(mountains_url, "mountains.xlsx")

capitals_output = process_excel(
    file_path="capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    output_path="capitals_enriched.xlsx",
    include_debug_columns=False,
)

mountains_output = process_excel(
    file_path="mountains.xlsx",
    task_description="додай висоту гір у метрах до колонки height",
    output_path="mountains_enriched.xlsx",
    include_debug_columns=False,
)

print(capitals_output)
print(mountains_output)

Enrichment plan:
{
  "task_type": "capital_distance_km",
  "target_column": "distance",
  "required_columns": [
    "Capital_From",
    "Capital_To"
  ],
  "value_unit": "km"
}


Resolving capitals:   0%|          | 0/14 [00:00<?, ?it/s]

Saved: capitals_enriched.xlsx
Enrichment plan:
{
  "task_type": "mountain_height_m",
  "target_column": "height",
  "required_columns": [
    "Mountain"
  ],
  "value_unit": "m"
}


Resolving mountains:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: mountains_enriched.xlsx
capitals_enriched.xlsx
mountains_enriched.xlsx


In [18]:
# Chunk 10: Preview and validate enriched Excel data
capitals_result = pd.read_excel("capitals_enriched.xlsx")
mountains_result = pd.read_excel("mountains_enriched.xlsx")

display(Markdown("## Capitals enriched"))
display(capitals_result)

display(Markdown("## Mountains enriched"))
display(mountains_result)

assert "distance_source" not in capitals_result.columns
assert "distance_status" not in capitals_result.columns
assert "height_source" not in mountains_result.columns
assert "height_status" not in mountains_result.columns

assert "distance" in capitals_result.columns
assert "height" in mountains_result.columns

assert capitals_result["distance"].notna().all()
assert mountains_result["height"].notna().all()

print("Validation passed: original structure preserved and target columns filled.")

## Capitals enriched

,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,2023
1,London,United Kingdom,Rome,Italy,1434
2,Paris,France,Berlin,Germany,877
3,Berlin,Germany,Vienna,Austria,524
4,Warsaw,Poland,Kyiv,Ukraine,689
5,Rome,Italy,Madrid,Spain,1364
6,Madrid,Spain,Lisbon,Portugal,502
7,Vienna,Austria,Budapest,Hungary,214
8,Stockholm,Sweden,Oslo,Norway,416
9,Athens,Greece,Istanbul,Turkey,562


## Mountains enriched

,Mountain,Country,height
0,Mount Everest,Nepal/China,8849
1,K2,Pakistan/China,8611
2,Kangchenjunga,Nepal/India,8586
3,Lhotse,Nepal/China,8516
4,Makalu,Nepal/China,8485
5,Cho Oyu,Nepal/China,8188
6,Dhaulagiri,Nepal,8167
7,Manaslu,Nepal,8163
8,Nanga Parbat,Pakistan,8126
9,Annapurna,Nepal,8091


Validation passed: original structure preserved and target columns filled.


In [19]:
# Chunk 11: Download enriched files
download_result_file("capitals_enriched.xlsx")
download_result_file("mountains_enriched.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>